# 🧱 第三课 · 区块解剖

## 本课目标

- 知道一个比特币区块里面到底装了什么
- 理解区块头 80 字节的 6 个字段各自干什么
- 理解 Merkle 树是什么、为什么需要它
- 能自己生成 Merkle Proof 证明一笔交易在区块中

> 这一课偏"数据结构"，但你会看到它和挖矿的直接关系。

## 1. 区块 = 区块头 + 交易列表

一个比特币区块分两部分：

```
┌─────────────────────────┐
│   区块头 (80 字节)       │  ← 矿工反复哈希的东西
│   - 版本号               │
│   - 前一区块哈希          │
│   - Merkle 根            │
│   - 时间戳               │
│   - nBits (难度目标)      │
│   - nonce (工作量证明)    │
├─────────────────────────┤
│   交易列表                │  ← 不用哈希，但被 Merkle 根代表
│   - Coinbase 交易         │
│   - 普通交易 × N          │
└─────────────────────────┘
```

**关键点**：矿工只哈希 80 字节的区块头，不直接哈希交易。
交易通过 Merkle 树被"压缩"成 32 字节的根，放在区块头里。
这样无论区块包含多少笔交易，矿工要处理的数据量不变。

## 2. 区块头 80 字节逐个看

| 字段 | 大小 | 类型 | 作用 |
|------|------|------|------|
| version | 4 字节 | 小端整数 | 区块格式版本号，当前是 1 或 2 |
| prev_block | 32 字节 | 字节串 | 前一区块的双 SHA-256 哈希 |
| merkle_root | 32 字节 | 字节串 | 所有交易的 Merkle 树根 |
| timestamp | 4 字节 | 小端整数 | Unix 时间戳（秒） |
| bits | 4 字节 | 小端整数 | nBits 压缩难度目标 |
| nonce | 4 字节 | 小端整数 | 矿工一直加 1，直到找到有效哈希 |

**总共刚好 80 字节。**

> 注意：比特币所有整数字段都是**小端序**（little-endian）。

In [ ]:
import hashlib, struct, time
from dataclasses import dataclass, field

@dataclass
class BlockHeader:
    version: int = 1
    prev_hash: bytes = field(default_factory=lambda: b"\x00" * 32)
    merkle_root: bytes = field(default_factory=lambda: b"\x00" * 32)
    timestamp: int = 0
    bits: int = 0x1d00ffff
    nonce: int = 0

    def serialize(self) -> bytes:
        # 格式: <I32s32sIII
        # < = 小端序, I = 4字节无符号整数, 32s = 32字节串
        return struct.pack(
            "<I32s32sIII",
            self.version,
            self.prev_hash,
            self.merkle_root,
            self.timestamp,
            self.bits,
            self.nonce,
        )

    def hash(self) -> bytes:
        # 双重 SHA-256，显示时反转（比特币惯例：大端显示）
        d = hashlib.sha256(hashlib.sha256(self.serialize()).digest()).digest()
        return d[::-1]  # 反转字节序

# 测试
header = BlockHeader(timestamp=int(time.time()))
print(f"序列化后大小: {len(header.serialize())} 字节 (应该是 80)")
print(f"区块哈希:     {header.hash().hex()}")

## 3. Merkle 树 — 把几千笔交易压成 32 字节

问题：一个区块可以包含 2000+ 笔交易，但区块头只有 32 字节留给"交易哈希"。

解法：用 Merkle 树把任意数量的交易递归哈希成**一个 32 字节的根**。

```
          MERKLE ROOT (H_ABCD)
         /                    \
     H_AB                     H_CD
    /    \                   /    \
  H_A    H_B              H_C    H_D
  /        \              /        \
TxA       TxB           TxC       TxD
```

**构造方法**：
1. 对每笔交易做双重 SHA-256 → 叶子节点
2. 相邻两个节点拼起来再哈希 → 上层节点
3. 重复直到只剩一个节点 → Merkle 根
4. 如果某一层节点数是奇数，复制最后一个

**为什么需要它**：
- **轻节点 (SPV)** 不需要下载整个区块。只需区块头 + Merkle Proof
- **矿工** 改 coinbase 时只需重算一条路径，不用重算整棵树
- **效率**：验证一笔交易只要 O(log n) 个哈希

In [ ]:
def hash256(data: bytes) -> bytes:
    # 比特币 Merkle 树节点的标准算法
    return hashlib.sha256(hashlib.sha256(data).digest()).digest()

def build_merkle_root(tx_list):
    # 输入: 交易数据列表（bytes）
    # 输出: 32 字节 Merkle 根
    if not tx_list:
        return b"\x00" * 32

    # 第一步：每笔交易做 hash256
    hashes = [hash256(tx) for tx in tx_list]

    # 第二步：逐层合并
    while len(hashes) > 1:
        if len(hashes) % 2 == 1:
            hashes.append(hashes[-1])  # 奇数：复制最后一个
        hashes = [
            hash256(hashes[i] + hashes[i+1])
            for i in range(0, len(hashes), 2)
        ]

    return hashes[0]

# 测试：8 笔交易 → 1 个根
txs = [f"Transaction_{i}".encode() for i in range(8)]
root = build_merkle_root(txs)
print(f"交易数: {len(txs)}")
print(f"Merkle 根: {root.hex()}")
print(f"大小:      {len(root)} 字节")
print(f"💡 8 笔交易 → 1 个 32 字节的根")  

## 4. Merkle Proof — "我证明这笔交易在区块里"

假设你是 SPV 钱包（手机钱包）。你不想下载整个 2MB 的区块，
但你想确认一笔交易确实被打包了。

Merkle Proof 就是你需要的"证明"：
- 给你的交易哈希
- 加上路径上每个兄弟节点的哈希
- 一路算到 Merkle 根
- 如果等于区块头里的 merkle_root → 交易确实在区块中

**证明大小 = O(log n)**。2000 笔交易的区块，证明只需约 11 个哈希（352 字节）。

下面我们来生成和验证一个 Merkle Proof：

In [ ]:
def generate_merkle_proof(tx_list, tx_index):
    # 输入: 交易列表, 目标交易索引
    # 输出: [(兄弟哈希, 是否在右边), ...]
    hashes = [hash256(tx) for tx in tx_list]
    proof = []
    idx = tx_index

    while len(hashes) > 1:
        if len(hashes) % 2 == 1:
            hashes.append(hashes[-1])

        if idx % 2 == 0:
            # 我是左节点 → 需要右兄弟
            proof.append((hashes[idx + 1], True))
        else:
            # 我是右节点 → 需要左兄弟
            proof.append((hashes[idx - 1], False))

        # 上层
        hashes = [hash256(hashes[i] + hashes[i+1])
                  for i in range(0, len(hashes), 2)]
        idx //= 2

    return proof

def verify_merkle_proof(tx, proof, merkle_root):
    # 从交易出发，沿 proof 一路算到根
    current = hash256(tx)
    for sibling, is_right in proof:
        if is_right:
            current = hash256(current + sibling)
        else:
            current = hash256(sibling + current)
    return current == merkle_root

# 完整演示
txs = [f"Tx_{i}".encode() for i in range(8)]
root = build_merkle_root(txs)
target = 3  # 要证明第 3 笔交易

proof = generate_merkle_proof(txs, target)

print(f"Merkle 根: {root.hex()}")
print(f"\n证明 Tx #{target} 在区块中:")
for i, (sibling, is_right) in enumerate(proof):
    side = "右" if is_right else "左"
    print(f"  第{i+1}步: 与{side}兄弟 {sibling.hex()[:16]}... 合并哈希")

result = verify_merkle_proof(txs[target], proof, root)
print(f"\n验证结果: {'✅ 通过' if result else '❌ 失败'}")
print(f"证明大小: {len(proof)} 个哈希 ({len(proof)*32} 字节)")
print(f"💡 不用下载 {len(txs)} 笔交易，只需 {len(proof)} 个哈希就能验证")  

## 5. Coinbase 交易 — 矿工的"工资单"

每个区块的**第一笔交易**永远是 coinbase 交易（也叫"创币交易"）：

- **没有输入**（不花任何人的币）
- **输出** = 区块奖励 + 所有交易手续费
- 有一个 **coinbase 数据字段**（最多 100 字节）可以写任意内容

中本聪在创世区块的 coinbase 里写了：

> The Times 03/Jan/2009 Chancellor on brink of second bailout for banks

这既是对传统银行体系的嘲讽，也是一个**不可篡改的时间戳**——
证明这个区块不可能在 2009 年 1 月 3 日之前产生。

**现代矿工的用法**：coinbase 的 extraNonce 字段用来扩展 nonce 搜索空间。
因为区块头 nonce 只有 32 位不够用，矿工每次改 extraNonce 就相当于
"换了一个搜索空间"，可以继续试。

## 6. 链 = 哈希指针链表

```
[Block 0] ←── [Block 1] ←── [Block 2] ←── [Block 3]
              prev_hash     prev_hash     prev_hash
              = H(Block0)   = H(Block1)   = H(Block2)
```

每一个区块里存了前一个区块的哈希。

**修改任何一个区块 → 它的哈希变 → 下一个区块的 prev_hash 对不上 → 链断了。**

这就是"区块链"三个字的来历：用哈希指针连接的不可篡改数据结构。

In [ ]:
# 演示：搭一条迷你链
import random

def mine_simple(header, target_zeros=3):
    target = b"\x00" * target_zeros
    for nonce in range(50_000_000):
        header.nonce = nonce
        h = hashlib.sha256(hashlib.sha256(header.serialize()).digest()).digest()
        if h.startswith(target):
            return True
    return False

chain = []
prev_hash = b"\x00" * 32

print("搭建一条 5 个区块的迷你链：\n")
for i in range(5):
    # 模拟随机交易
    txs = [f"tx_{i}_{j}".encode() for j in range(random.randint(2, 4))]
    merkle = build_merkle_root(txs)

    header = BlockHeader(
        prev_hash=prev_hash,
        merkle_root=merkle,
        timestamp=int(time.time()),
    )

    ok = mine_simple(header, target_zeros=3)
    block_hash = header.hash() if ok else b"FAIL"
    chain.append((header, block_hash))
    prev_hash = block_hash

    if ok:
        print(f"  Block {i}: ✅ nonce={header.nonce}, hash={block_hash.hex()[:16]}...")
    else:
        print(f"  Block {i}: ❌ 挖矿失败")

# 验证链完整性
valid = all(
    chain[i][1] == chain[i+1][0].prev_hash
    for i in range(len(chain) - 1)
)
print(f"\n链完整性: {'✅ 完整' if valid else '❌ 断裂'}")
print("💡 每个区块的哈希 = 下一个区块的 prev_hash")  

## 本课小结

| 概念 | 记住它 |
|------|--------|
| 区块头 | 80 字节，6 个字段，矿工只哈希这个 |
| Merkle 树 | 把 N 笔交易压缩成 32 字节的根 |
| Merkle Proof | 用 O(log n) 数据证明交易在区块中 |
| Coinbase | 第一笔交易，矿工自己给自己发工资 |
| 链 | 每个区块指向前一个，改了历史全断 |

**下一课**：把所有零件拼起来，亲手实现一个完整的矿工程序。